# evolving-graphrag — results analysis

Reads `results/summary.json` (written by `make eval`) and reproduces every figure and
table in the report. Nothing is computed here that is not also computed by the harness —
this notebook is a *view* onto a run, so a number in the write-up and a number here can
never drift apart.

```bash
make eval          # regenerate results/ from scratch
```

Every run carries a `config_hash` derived from the whole settings object; quote it beside
any figure you lift out of here.

In [ ]:
import json
from pathlib import Path

import pandas as pd

RESULTS = Path("../results")
summary = json.loads((RESULTS / "summary.json").read_text())

print(f"config_hash : {summary['config_hash']}")
print(f"seed        : {summary['seed']}")
print(f"version     : {summary['version']}")
print(f"wall clock  : {summary['wall_s']}s")
print(f"environment : {summary['environment']}")
print(f"benchmark   : {summary['benchmark']}")

## 1. The headline table — deletion

`stale_answer_rate` is the fraction of questions still answered with a fact that **only a
deleted document supported**. `deletion-survivors` is the control: facts a *surviving*
document still supports must remain answerable, so a system cannot win by deleting too
much.

In [ ]:
scenarios = pd.DataFrame(
    [
        {
            "system": s["system"],
            "scenario": s["scenario"],
            "f1": s["f1"],
            "recall": s["recall"],
            "stale_answer_rate": s["stale_answer_rate"],
            "abstention_rate": s["abstention_rate"],
            "update_tokens": s["update_cost"]["tokens"],
            "update_wall_ms": s["update_cost"]["wall_ms"],
            "recomputed": s["update_cost"]["communities_recomputed"],
            "entities": s["index"]["entities"],
            "relations": s["index"]["relations"],
        }
        for s in summary["scenarios"]
    ]
)

deletion = scenarios[scenarios.scenario.isin(["deletion", "deletion-survivors"])]
deletion.pivot(index="system", columns="scenario", values=["stale_answer_rate", "recall"])

In [ ]:
# Cost of keeping up, by system and phase.
scenarios.pivot(index="system", columns="scenario", values="update_tokens")

## 2. Cost vs freshness

The knob is the **recompute budget**: how many dirty community summaries may be
regenerated per document change.

Note which freshness measure moves. `stale_answer_rate` is a property of the *graph* —
once provenance GC is correct it is 0 at every budget, because retraction does not depend
on summaries. `mean_stale_touched` is what the *reader* sees, and it is what the budget
actually buys.

In [ ]:
pareto = pd.DataFrame(summary["pareto"]["points"])
print("Pareto frontier:", summary["pareto"]["frontier"])
pareto[
    [
        "label",
        "update_tokens",
        "update_usd",
        "communities_recomputed",
        "dirty_fraction_after",
        "mean_stale_touched",
        "stale_answer_rate",
        "base_recall",
        "survivor_recall",
    ]
]

In [ ]:
reference = pareto[pareto.label == "full-reindex"].update_tokens.iloc[0]
fresh = pareto[(pareto.label.str.startswith("evolving")) & (pareto.mean_stale_touched == 0)]
cheapest_fresh = fresh.update_tokens.min()
cheapest = pareto[pareto.label.str.startswith("evolving")].update_tokens.min()

print(f"full reindex                     : {reference:,} tokens")
print(f"cheapest fully-fresh incremental : {cheapest_fresh:,} tokens"
      f"  ({reference / cheapest_fresh:.1f}x cheaper, same freshness)")
print(f"cheapest incremental overall     : {cheapest:,} tokens"
      f"  ({reference / cheapest:.1f}x cheaper, some staleness accepted)")

## 3. Scaling — the asymptotic argument

The cost of one document change should stay **flat** as the corpus grows, while a rebuild
grows with it. That ratio, not any single number, is the argument for incremental
maintenance.

In [ ]:
scaling = pd.DataFrame(summary["scaling"])
display(scaling)

spread = scaling.one_update_tokens.max() / scaling.one_update_tokens.min()
print(f"\none-update cost varies by {spread:.2f}x across a {scaling.documents.max() // scaling.documents.min()}x corpus")
print(f"rebuild/update ratio grows {scaling.reindex_cost_ratio.iloc[0]}x -> {scaling.reindex_cost_ratio.iloc[-1]}x")

In [ ]:
# Memory growth vs corpus size.
scaling[["documents", "chunks", "entities", "relations", "communities"]].set_index("documents")

## 4. Compaction ablation

Incremental placement drifts — a chain of local decisions does not reproduce a global
clustering. This measures whether the drift matters and whether periodic re-clustering
pays for itself. `compaction_interval = None` means compaction never ran.

In [ ]:
pd.DataFrame(summary["compaction"])

## 5. Entity-resolution threshold sweep

The risk register's mitigation, measured. Too low a `tau_sim` fuses distinct entities —
and a bad merge is worse than a duplicate here, because merging fuses two provenance sets
and a later deletion can no longer separate them. Too high leaves duplicates behind and
fragments the graph.

In [ ]:
resolution = pd.DataFrame(summary["resolution"])
display(resolution)
print(f"\nconfigured tau_sim sits at the elbow: entities {resolution.entities.min()}"
      f"–{resolution.entities.max()} across the sweep")

## 6. Per-question detail

Where a scenario score came from. Useful when a number looks surprising: the prediction,
the citations, and whether a stale fact leaked are all recorded per question.

In [ ]:
rows = [
    {"system": s["system"], "scenario": s["scenario"], **q}
    for s in summary["scenarios"]
    for q in s["per_question"]
]
detail = pd.DataFrame(rows)

# Every stale answer in the run, with the system that produced it.
detail[detail.stale][["system", "scenario", "question", "prediction"]]

In [ ]:
# Side-by-side on the deletion scenario: what each system said after the delete.
pd.set_option("display.max_colwidth", 90)
detail[detail.scenario == "deletion"].pivot(
    index="question", columns="system", values="prediction"
)

## 7. Figures

`make figures` writes these as SVG next to the CSVs. They are drawn from exactly the rows
above.

In [ ]:
from IPython.display import SVG, display

for name in ("fig_pareto.svg", "fig_scaling.svg", "fig_deletion.svg"):
    path = RESULTS / name
    if path.exists():
        display(SVG(filename=str(path)))
    else:
        print(f"{name} missing — run `make figures`")